# 🚀 BẢN 2: HUẤN LUYỆN & ĐÁNH GIÁ MÔ HÌNH CẢI TIẾN (OURS)
### 📘 Hard Instance-Aware Multi-Agent Perception with 5 Proposed Enhancements

Notebook này dùng để huấn luyện và kiểm thử **Mô hình đề xuất cải tiến của đề tài (FocalComm-Ours)**:
1. **DINOv3-Enhanced Feature Extractor:** Tăng cường trích xuất đặc trưng không gian BEV đa tỉ lệ, làm rõ các vật thể nhỏ và bị che khuất.
2. **MLP Projector:** Ánh xạ phi tuyến 3 tầng thay thế phép chiếu tuyến tính đơn giản.
3. **Semantic-Guided Masking:** Mặt nạ ngữ nghĩa phân cấp độ ưu tiên, tập trung tối đa vào người đi bộ (VRU) và góc khuất.
4. **QDAFF (Query Diffusion-guided Adaptive Feature Fusion):** Hợp nhất đặc trưng thích ứng tích hợp diffusion khử nhiễu truyền thông.
5. **FHIM (FAISS Hard Instance Mining):** Tăng tốc độ khai phá và đánh chỉ mục vector tương đồng các đối tượng khó giữa các xe.

---
### 🎯 Mục Tiêu Nghiên Cứu:
Vượt qua mốc chuẩn của Baseline (56.1% mAP), đặc biệt tối ưu độ chính xác ở lớp khó (Pedestrian/VRU) và tiết kiệm băng thông giao tiếp.

## 1. Kiểm tra môi trường GPU trên Kaggle

In [ ]:
import os
import sys
import torch

print(f"Python Version: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ CẢNH BÁO: Bật GPU trong Kaggle settings!")

!nvidia-smi


## 2. Clone Repository FocalComm từ GitHub
Lấy mã nguồn chứa đầy đủ mô hình cải tiến FocalComm-Ours.

In [ ]:
!rm -rf /kaggle/working/FocalComm
!git clone https://github.com/tranvuquanghuy12/FocalComm.git /kaggle/working/FocalComm

%cd /kaggle/working/FocalComm
!git log -n 1 --oneline


## 3. Cài đặt các thư viện cần thiết & Biên dịch CUDA Extensions

In [ ]:
# 1. Thư viện cơ bản
!pip install -q easydict numba open3d cython tensorboardX shapely==2.0.0
!pip install -q pyquaternion einops timm icecream python-lzf termcolor efficientnet_pytorch==0.7.0 huggingface_hub
!pip install -q git+https://github.com/klintan/pypcd.git

# 2. Cài spconv
!pip install -q spconv-cu118 || pip install -q spconv-cu120 || pip install -q spconv-cu117

# 3. Biên dịch CUDA & Cython Extensions
%cd /kaggle/working/FocalComm/focalcomm/utils
!python setup.py build_ext --inplace

%cd /kaggle/working/FocalComm/focalcomm/pcdet_utils
!python setup.py build_ext --inplace

%cd /kaggle/working/FocalComm
print("✅ Môi trường cho mô hình cải tiến đã sẵn sàng!")


## 4. Thiết lập Bộ Dữ Liệu V2X-Real

In [ ]:
import os
DATASET_ROOT = "/kaggle/working/dataset/V2XReal"
os.makedirs(DATASET_ROOT, exist_ok=True)

kaggle_input_dirs = [d for d in os.listdir("/kaggle/input") if "v2x" in d.lower() or "focalcomm" in d.lower()] if os.path.exists("/kaggle/input") else []
if kaggle_input_dirs:
    src = os.path.join("/kaggle/input", kaggle_input_dirs[0])
    print(f"Phát hiện dataset trong Kaggle Input: {src}")
    if not os.path.exists(os.path.join(DATASET_ROOT, "train")) and os.path.exists(os.path.join(src, "train")):
        os.symlink(os.path.join(src, "train"), os.path.join(DATASET_ROOT, "train"))
    if not os.path.exists(os.path.join(DATASET_ROOT, "val")) and os.path.exists(os.path.join(src, "val")):
        os.symlink(os.path.join(src, "val"), os.path.join(DATASET_ROOT, "val"))
    print(f"Liên kết dữ liệu thành công vào: {DATASET_ROOT}")
else:
    print(f"Dataset sẽ được đọc từ: {DATASET_ROOT}")


## 5. Khởi Chạy Huấn Luyện Mô Hình Cải Tiến (Ours Training)
Mô hình sử dụng file cấu hình `focalcomm_ours_kaggle.yaml` kích hoạt toàn bộ 5 cải tiến:
- `DinoV3BEVEnhancer`
- `MLPProjector`
- `SemanticInstanceMasker`
- `QDAFFV3` (Query Diffusion-guided Fusion)
- `FAISSHardInstanceMiner` (FHIM)

In [ ]:
# Huấn luyện mô hình cải tiến FocalComm-Ours
# Sử dụng flag --subset nếu muốn kiểm tra dry-run 1-2 kịch bản trước
!python focalcomm/tools/train_focalcomm.py \
    --hypes_yaml focalcomm/hypes_yaml/v2xreal/focalcomm_ours_kaggle.yaml


## 6. Đánh Giá Độ Chính Xác (Inference & Evaluation)
Tính toán chi tiết các chỉ số AP@0.3, AP@0.5, AP@0.7 trên từng lớp (Car, Pedestrian, Truck) và mAP tổng thể.

In [ ]:
# Đánh giá checkpoint tốt nhất của mô hình cải tiến
import glob

saved_models = glob.glob("focalcomm/logs/vr_fcm_ours_kaggle_gpu*/**/net_epoch_best.pth", recursive=True)
model_dir = os.path.dirname(saved_models[0]) if saved_models else "focalcomm/logs/vr_fcm_ours_kaggle_gpu"

print(f"Đang đánh giá mô hình cải tiến tại: {model_dir}")
!python focalcomm/tools/inference.py \
    --model_dir {model_dir} \
    --fusion_method intermediate


## 7. Đóng Gói Checkpoints và Kết Quả Tải Về

In [ ]:
output_tar = "/kaggle/working/ours_focalcomm_results.tar.gz"
!tar -czvf {output_tar} focalcomm/logs/ checkpoints/
print(f"✅ Đã tạo file kết quả mô hình cải tiến tại: {output_tar}")
